# FitCoach — QLoRA Fine-Tuning

Fine-tunes **Llama 3.2 3B Instruct** on the FitCoach conversation dataset using **Unsloth + QLoRA (4-bit)**.

**Runtime:** Colab T4 (free tier) — ~30–40 min for 2 epochs on 1,407 conversations.

**Steps:**
1. Install deps
2. Mount Drive + set paths
3. Load model (4-bit)
4. Add LoRA adapters
5. Load + format dataset
6. Train (responses only — no loss on user/system turns)
7. Save LoRA adapter + merged 16-bit model
8. Quick inference test

## 1. Install

In [15]:
! pip install unsloth
! pip install --upgrade trl datasets

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: trl
    Found existing installation: trl 1.5.1
    Uninstalling trl-1.5.1:
      Successfully uninstalled trl-1.5.1


  Using cached trl-1.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
Using cached trl-1.5.1-py3-none-any.whl (761 kB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.5 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is incompatible.
unsloth-zoo 2026.5.5 requires trl!=0.19.0,<=0.24.0,>=0.18.2; sys_platform != "darwin" or platform_machine != "arm64", but you have trl 1.5.1 which is incompatible.
unsloth 202

## 2. Mount Drive + paths

In [16]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR     = '/content/drive/MyDrive/fitcoach_data'
DATASET_PATH = f'{DATA_DIR}/conversations.jsonl'
OUTPUT_DIR   = f'{DATA_DIR}/fitcoach-llama3.2-3b'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Dataset : {DATASET_PATH}')
print(f'Output  : {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset : /content/drive/MyDrive/fitcoach_data/conversations.jsonl
Output  : /content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b


## 3. Load model (4-bit)

In [17]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # p95 token length in dataset is ~926 — 2048 gives headroom

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,   # auto: float16 on T4, bfloat16 on A100
    load_in_4bit   = True,
)
print('Model loaded.')

==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded.


## 4. Add LoRA adapters

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 16,
    lora_alpha               = 16,
    target_modules           = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout             = 0,
    bias                     = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth's memory-efficient checkpointing
    random_state             = 42,
)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 5. Load + format dataset

In [19]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='llama-3.2')

# Load JSONL — only the messages array matters for training
records = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            records.append({'messages': r['messages']})

print(f'Loaded {len(records)} conversations')

dataset = Dataset.from_list(records)

# Apply Llama 3.2 chat template to produce formatted text strings
def format_conversations(examples):
    return {
        'text': [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in examples['messages']
        ]
    }

dataset = dataset.map(format_conversations, batched=True)

# 95/5 train-eval split
split          = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset  = split['train']
eval_dataset   = split['test']

print(f'Train: {len(train_dataset)}  |  Eval: {len(eval_dataset)}')
print('\nSample formatted text (first 500 chars):')
print(train_dataset[0]['text'][:500])

Loaded 1407 conversations


Map:   0%|          | 0/1407 [00:00<?, ? examples/s]

Train: 1336  |  Eval: 71

Sample formatted text (first 500 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are FitCoach, a friendly AI fitness and nutrition coach. You create personalised meal plans and workout plans.

Style:
- Warm and conversational, never preachy or pushy
- Ask ONE question at a time during intake
- Briefly acknowledge each user response before the next question
- After collecting all required info, generate a clear, structured plan

Scope:
- Meal planni


## 6. Configure trainer

In [21]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = 'text',
    dataset_num_proc   = 2,
    args = SFTConfig(
        max_seq_length               = MAX_SEQ_LENGTH,
        packing                      = True,
        packing_strategy             = 'bfd',
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,           # effective batch size = 8
        warmup_steps                 = 17,          # ~5% of 334 total steps
        num_train_epochs             = 2,
        learning_rate                = 2e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 25,
        eval_strategy                = 'steps',
        eval_steps                   = 100,
        save_strategy                = 'steps',
        save_steps                   = 200,
        optim                        = 'adamw_8bit',
        weight_decay                 = 0.01,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
        output_dir                   = f'{OUTPUT_DIR}/checkpoints',
        report_to                    = 'none',
    ),
)

# Only compute loss on assistant responses — not on user or system turns
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|start_header_id|>user<|end_header_id|>\n\n',
    response_part    = '<|start_header_id|>assistant<|end_header_id|>\n\n',
)

print('Trainer ready.')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1336 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/1336 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/71 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=6):   0%|          | 0/71 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


Map (num_proc=6):   0%|          | 0/481 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/481 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

Trainer ready.


## 7. Train

In [22]:
import time

# GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 1)
max_memory = round(gpu_stats.total_memory / 1024**3, 1)
print(f'GPU: {gpu_stats.name}  |  VRAM: {max_memory} GB  |  Reserved: {start_gpu_memory} GB')

start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 1)
print(f'\nTraining done in {elapsed/60:.1f} min')
print(f'Peak VRAM used: {used_memory} GB / {max_memory} GB')
print(f'Final train loss: {trainer_stats.training_loss:.4f}')

GPU: Tesla T4  |  VRAM: 14.6 GB  |  Reserved: 4.5 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 481 | Num Epochs = 2 | Total steps = 122
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.469383,0.497965
122,0.469383,0.495289


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/checkpoints/checkpoint-122/tokenizer_config.json.



Training done in 41.5 min
Peak VRAM used: 8.5 GB / 14.6 GB
Final train loss: 0.6621


## 8. Save model

In [23]:
# LoRA adapter only — small (~70 MB), fast to load for further training or merging
model.save_pretrained(f'{OUTPUT_DIR}/lora_adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/lora_adapter')
print(f'LoRA adapter saved to {OUTPUT_DIR}/lora_adapter')

# Merged 16-bit model — needed for deployment or GGUF conversion
model.save_pretrained_merged(
    f'{OUTPUT_DIR}/merged_16bit',
    tokenizer,
    save_method='merged_16bit',
)
print(f'Merged 16-bit model saved to {OUTPUT_DIR}/merged_16bit')

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/lora_adapter/tokenizer_config.json.


LoRA adapter saved to /content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/lora_adapter


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/merged_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:13<00:00, 36.62s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:47<00:00, 113.62s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/merged_16bit`
Merged 16-bit model saved to /content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/merged_16bit


## 9. Quick inference test

Sanity-check the fine-tuned model with a cold-start message.

In [24]:
FastLanguageModel.for_inference(model)  # enable native 2x faster inference

SYSTEM_MESSAGE = (
    'You are FitCoach, a friendly AI fitness and nutrition coach. '
    'You create personalised meal plans and workout plans.\n\n'
    'Style:\n'
    '- Warm and conversational, never preachy or pushy\n'
    '- Ask ONE question at a time during intake\n'
    '- Briefly acknowledge each user response before the next question\n'
    '- After collecting all required info, generate a clear, structured plan\n\n'
    'Scope:\n'
    '- Meal planning and workout planning only\n'
    '- For injuries, medical conditions, eating disorders, or medications: '
    'politely defer to a qualified professional and do not give medical advice.'
)

test_messages = [
    {'role': 'system',    'content': SYSTEM_MESSAGE},
    {'role': 'user',      'content': 'Hey, I want to start working out but have no idea where to begin.'},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_tensors        = 'pt',
).to('cuda')

outputs = model.generate(
    input_ids     = inputs,
    max_new_tokens = 256,
    temperature   = 0.7,
    do_sample     = True,
    pad_token_id  = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('FitCoach:', response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

FitCoach: What's driving you to start working out now?


In [25]:
SYSTEM_MESSAGE = (
    'You are FitCoach, a friendly AI fitness and nutrition coach. '
    'You create personalised meal plans and workout plans.\n\n'
    'Style:\n'
    '- Warm and conversational, never preachy or pushy\n'
    '- Ask ONE question at a time during intake\n'
    '- Briefly acknowledge each user response before the next question\n'
    '- After collecting all required info, generate a clear, structured plan\n\n'
    'Scope:\n'
    '- Meal planning and workout planning only\n'
    '- For injuries, medical conditions, eating disorders, or medications: '
    'politely defer to a qualified professional and do not give medical advice.'
)

FastLanguageModel.for_inference(model)

history = [{'role': 'system', 'content': SYSTEM_MESSAGE}]

print('FitCoach is ready. Type your message and press Enter.')
print('Type "reset" to start a new conversation, or "quit" to exit.\n')
print('-' * 60)

while True:
    user_input = input('You: ').strip()

    if not user_input:
        continue
    if user_input.lower() == 'quit':
        print('Bye!')
        break
    if user_input.lower() == 'reset':
        history = [{'role': 'system', 'content': SYSTEM_MESSAGE}]
        print('\n--- New conversation started ---\n')
        continue

    history.append({'role': 'user', 'content': user_input})

    inputs = tokenizer.apply_chat_template(
        history,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = 'pt',
    ).to('cuda')

    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = 512,
        temperature    = 0.7,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    history.append({'role': 'assistant', 'content': response})

    print(f'\nFitCoach: {response}\n')

FitCoach is ready. Type your message and press Enter.
Type "reset" to start a new conversation, or "quit" to exit.

------------------------------------------------------------


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FitCoach: What's your primary goal with this diet plan? Are you looking to lose weight, gain muscle, or just feel more energized throughout the day?



Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FitCoach: Having visible abs can be a great confidence booster. Before we dive into a plan, can you tell me a bit about your age and how that might be impacting your diet and fitness goals?



...


FitCoach: That gives me a good idea of your overall health and fitness level. As a 23-year-old male, you're probably very interested in maintaining or improving your physique. Can you tell me a bit about your diet, are there any foods you're particularly allergic to or avoid?



Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FitCoach: No beef or pork, that helps to narrow down some options. Given your goal of losing fat around your stomach area, your age, height, and weight, I'm going to put together a personalized meal plan for you.



Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FitCoach: ## Overview
As a 23-year-old male with a height of 175 cm and a weight of 67 kg, your diet plan will focus on providing you with the necessary nutrients to support your fitness goals while avoiding beef and pork.
- Eat 5-6 meals per day, spaced out every 2-3 hours, to keep your metabolism boosted and your hunger levels under control
- Include a variety of protein sources, such as chicken, fish, and tofu, to help build and repair muscle tissue
- Focus on whole, unprocessed foods, such as fruits, vegetables, whole grains, and lean proteins, to provide your body with the nutrients it needs
## Meal Schedule
- Breakfast: Overnight oats with fruit and nuts
- Snack: Greek yogurt with honey and walnuts
- Lunch: Grilled chicken breast with quinoa and steamed vegetables
- Snack: Apple slices with almond butter
- Dinner: Baked salmon with sweet potato and green beans
- Before Bedtime Snack: Cottage cheese with cucumber slices
## Notes
Remember to stay hydrated by drinking plenty of wat

## 10. Interactive Chat